# FraudShield -- sklearn Training (SageMaker Studio)

This notebook runs the same training logic as `train.py` (Script Mode) but interactively inside SageMaker Studio.

**Purpose:** Let you step through the training code cell by cell on SageMaker's managed infrastructure, so you can see exactly what happens at each stage before submitting it as a training job.

**Prerequisites:**
- SageMaker Studio Domain is **InService** (Step 2)
- Training data (`train.csv`) has been uploaded to S3 (Step 7 of the lecture notebook)
- This notebook is uploaded to SageMaker Studio (instructions below)

In [ ]:
%pip install -q numpy pandas scikit-learn joblib boto3 sagemaker

## 1. Environment Setup

When running inside SageMaker Studio, the execution role and S3 bucket are automatically available.

In [ ]:
import os
import boto3
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
from sagemaker_core.helper.session_helper import Session, get_execution_role

session = Session()
role = get_execution_role()
bucket = session.default_bucket()
region = session.boto_region_name

print(f"Role:    {role}")
print(f"Bucket:  {bucket}")
print(f"Region:  {region}")

## 2. Download Training Data from S3

In a managed training job, SageMaker downloads data into `/opt/ml/input/data/train/` automatically. Here we do it ourselves so we can inspect it.

In [ ]:
s3 = boto3.client("s3")

local_data_dir = "data"
os.makedirs(local_data_dir, exist_ok=True)

s3_key = "fraudshield/data/train/train.csv"
local_path = os.path.join(local_data_dir, "train.csv")

s3.download_file(bucket, s3_key, local_path)
print(f"Downloaded s3://{bucket}/{s3_key} -> {local_path}")

data = pd.read_csv(local_path)
print(f"\nShape: {data.shape}")
data.head()

## 3. Prepare Features and Target

Same split as `train.py` -- everything except the `target` column is a feature.

In [ ]:
X = data.drop("target", axis=1)
y = data["target"]

print(f"Features: {list(X.columns)}")
print(f"Target distribution:\n{y.value_counts()}")

## 4. Train the Model

Same hyperparameters as the Script Mode version: `n_estimators=100`, `random_state=42`.

In [ ]:
n_estimators = 100
random_state = 42

model = RandomForestClassifier(
    n_estimators=n_estimators,
    random_state=random_state,
)
model.fit(X, y)

accuracy = accuracy_score(y, model.predict(X))
print(f"Training accuracy: {accuracy:.4f}")

## 5. Save the Model Artifact

In a managed training job, SageMaker expects artifacts in `/opt/ml/model/`. Here we save locally and then upload to S3 to mirror that behavior.

In [ ]:
model_dir = "model"
os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "model.pkl")
joblib.dump(model, model_path)
print(f"Model saved locally: {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1024:.1f} KB")

## 6. Upload Model to S3

Mirror what SageMaker does at the end of a training job -- compress and upload the artifact.

In [ ]:
import tarfile

tar_path = "model.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(model_path, arcname="model.pkl")

s3_model_key = "fraudshield/output/studio-model/model.tar.gz"
s3.upload_file(tar_path, bucket, s3_model_key)
print(f"Model artifact uploaded to: s3://{bucket}/{s3_model_key}")

## Comparison: Studio Notebook vs. Managed Training Job

| Aspect | This Notebook (Studio) | Managed Training Job (Step 9) |
|--------|----------------------|------------------------------|
| Where it runs | Studio notebook instance | Dedicated training instance |
| Data download | Manual (`boto3`) | Automatic (data channels) |
| Model save | Manual (`joblib` + `boto3`) | Automatic (`/opt/ml/model/`) |
| Hyperparameters | Python variables | CLI args via `argparse` |
| Billing | Studio instance (always on while open) | Training instance (only during job) |
| Reproducibility | Depends on notebook state | Fully reproducible (immutable job record) |

**Takeaway:** Interactive notebooks are great for exploration and debugging. Managed training jobs are better for production workloads because they are reproducible, cost-efficient, and automatically tracked.